# Пример: экспоненты пути P5 и цикла C4

Здесь результат вычисления — **символьная полиформа**: видны её формы, коэффициенты и грейды. Сначала создаются элементы `a, b, c, d, e`, затем формы, лапласианы и экспоненты. Числа обозначают коэффициенты и номера грейдов, а не вершины.

## Подготовка

Выполните ячейки сверху вниз. Используется `polyform_algebra 0.9.0`.

In [1]:
from importlib.metadata import version, PackageNotFoundError
import subprocess
import sys

try:
    installed = version("polyform_algebra")
except PackageNotFoundError:
    installed = None
if installed != "0.9.0":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "https://github.com/magin67/polyform_algebra/archive/refs/tags/v0.9.0.zip"])
import polyform_algebra
assert polyform_algebra.get_version() == "0.9.0"
print("polyform_algebra", polyform_algebra.get_version())

polyform_algebra 0.9.0


## Элементы и формы

Переменная `e` ниже обозначает вершину; `unit` — единицу алгебры $\mathrm{e}$. Форма `f_ab` — это $[(ab)]^2$, где $(ab)=b-a$. В штатном выводе пакета `f[(a, b)]` означает эту же квадратичную форму, а `f[1]` — единицу алгебры.

In [2]:
from fractions import Fraction
from math import factorial
from polyform_algebra import Point, Polyform, quForm

a, b, c, d, e = Point.create_list("abcde")
f_ab = quForm([(a, b)])
f_bc = quForm([(b, c)])
f_cd = quForm([(c, d)])
f_de = quForm([(d, e)])
f_ad = quForm([(a, d)])
unit = Polyform.one()
print("vertices:", a, b, c, d, e)
print("f_ab =", f_ab)

vertices: a b c d e
f_ab = f[(a, b)]


## Умножение: слияние, раздельные компоненты, обнуление

$$[(ab)]^2[(bc)]^2=[(abc)]^2,\qquad [(ab)]^2[(cd)]^2=[(ab)(cd)]^2.$$

Общая вершина сливает две границы. При отсутствии общей вершины остаются две компоненты. Повторное ребро или замкнутый цикл даёт нулевое произведение.

In [3]:
merged = f_ab * f_bc
separate = f_ab * f_cd
print("f_ab * f_bc =", merged)
print("f_ab * f_cd =", separate)
print("f_ab * f_ab =", f_ab * f_ab)
print("f_ab * f_bc * f_ad * f_cd =", f_ab * f_bc * f_ad * f_cd)
assert merged.as_polyform().is_equivalent(quForm([(a, b, c)]).as_polyform())
assert separate.as_polyform().is_equivalent(quForm([(a, b), (c, d)]).as_polyform())
assert (f_ab * f_ab).is_zero()
assert (f_ab * f_bc * f_cd * f_ad).is_zero()

f_ab * f_bc = f[(a, b, c)]
f_ab * f_cd = f[(a, b), (c, d)]
f_ab * f_ab = f[]
f_ab * f_bc * f_ad * f_cd = f[]


### Вывод в обозначениях заметок

Следующая вспомогательная функция только оформляет уже вычисленные формы. Умножение и экспоненту выполняет пакет. Формулы ниже строятся из результата, а не подставляются готовыми строками.

In [4]:
from IPython.display import display, Math

def form_tex(form):
    # This display adapter only formats the computed boundary forms.
    if form.is_zero():
        return "0"
    if form.is_one():
        return r"\mathrm{e}"
    components = []
    for component in form.components:
        if not isinstance(component, tuple):
            raise TypeError("This tutorial formats boundary forms only")
        labels = [str(vertex) for vertex in component]
        if not all(label in "abcde" and len(label) == 1 for label in labels):
            raise ValueError("Expected the named tutorial vertices")
        components.append("(" + "".join(labels) + ")")
    return "[" + "".join(components) + "]^2"

def show(name, value):
    terms = value.terms if isinstance(value, Polyform) else {value: 1}
    pieces = []
    for form, coefficient in sorted(terms.items(), key=lambda item: str(item[0])):
        if coefficient == 0 or form.is_zero():
            continue
        magnitude = Fraction(abs(coefficient))
        scalar = (str(magnitude.numerator) if magnitude.denominator == 1 else
                  rf"\frac{{{magnitude.numerator}}}{{{magnitude.denominator}}}")
        body = form_tex(form)
        if magnitude != 1:
            body = scalar + r"\," + body
        sign = "-" if coefficient < 0 else ("+" if pieces else "")
        pieces.append(sign + body)
    if not pieces:
        pieces = ["0"]
    rows = [" ".join(pieces[i:i+3]) for i in range(0, len(pieces), 3)]
    display(Math(r"\begin{aligned}& " + name + " = " +
                 r"\\ &\quad ".join(rows) + r"\end{aligned}"))

In [5]:
show(r"[(ab)]^2[(bc)]^2", merged)
show(r"[(ab)]^2[(cd)]^2", separate)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Путь P5: полный символьный вывод

$a-b-c-d-e$. Все проводимости равны единице. `print(M_path)` сам группирует экспоненту по грейдам: `g0` — единица, `g1` — рёберные формы, `g2` — произведения двух рёбер и так далее.

In [6]:
L_path = f_ab + f_bc + f_cd + f_de
M_path = L_path.exp()
print("L_path =", L_path)
print("exp(L_path):")
print(M_path)

L_path = f[(a, b)] + f[(b, c)] + f[(c, d)] + f[(d, e)]
exp(L_path):
g0: f[1]
g1: f[(d, e)] + f[(c, d)] + f[(b, c)] + f[(a, b)]
g2: f[(c, d, e)] + f[(b, c), (d, e)] + f[(b, c, d)] + f[(a, b), (d, e)] + f[(a, b), (c, d)] + f[(a, b, c)]
g3: f[(b, c, d, e)] + f[(a, b), (c, d, e)] + f[(a, b, c), (d, e)] + f[(a, b, c, d)]
g4: f[(a, b, c, d, e)]


In [7]:
show(r"L_{P_5}", L_path)
for grade in range(5):
    show(rf"(M_{{P_5}})_{{{grade}}}", M_path.extract_grade(grade))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Как появляются грейды

$$\exp L=\prod_{xy\in E}(\mathrm{e}+f_{xy}),\qquad M_k=\frac{L^k}{k!}.$$

Каждый следующий множитель добавляет варианты с очередным ребром. Ниже видны все промежуточные полиформы, а не только старший коэффициент.

In [8]:
partial = unit
for edge in (f_ab, f_bc, f_cd, f_de):
    partial = partial * (unit + edge)
    print("Factor:", unit + edge)
    print(partial)
    print()
assert partial.is_equivalent(M_path)

Factor: g0: f[1]
g1: f[(a, b)]
g0: f[1]
g1: f[(a, b)]

Factor: g0: f[1]
g1: f[(b, c)]
g0: f[1]
g1: f[(b, c)] + f[(a, b)]
g2: f[(a, b, c)]

Factor: g0: f[1]
g1: f[(c, d)]
g0: f[1]
g1: f[(c, d)] + f[(b, c)] + f[(a, b)]
g2: f[(b, c, d)] + f[(a, b), (c, d)] + f[(a, b, c)]
g3: f[(a, b, c, d)]

Factor: g0: f[1]
g1: f[(d, e)]
g0: f[1]
g1: f[(d, e)] + f[(c, d)] + f[(b, c)] + f[(a, b)]
g2: f[(c, d, e)] + f[(b, c), (d, e)] + f[(b, c, d)] + f[(a, b), (d, e)] + f[(a, b), (c, d)] + f[(a, b, c)]
g3: f[(b, c, d, e)] + f[(a, b), (c, d, e)] + f[(a, b, c), (d, e)] + f[(a, b, c, d)]
g4: f[(a, b, c, d, e)]



## Цикл C4: четыре дерева, одна форма

$a-b-c-d-a$. Любые три ребра дают форму $[(abcd)]^2$. Все четыре ребра образуют цикл и дают ноль. Поэтому старший ненулевой грейд равен $4[(abcd)]^2$.

In [9]:
L_cycle = f_ab + f_bc + f_cd + f_ad
M_cycle = L_cycle.exp()
print("L_cycle =", L_cycle)
print("exp(L_cycle):")
print(M_cycle)

L_cycle = f[(a, b)] + f[(b, c)] + f[(c, d)] + f[(a, d)]
exp(L_cycle):
g0: f[1]
g1: f[(a, d)] + f[(c, d)] + f[(b, c)] + f[(a, b)]
g2: f[(a, c, d)] + f[(a, d), (b, c)] + f[(b, c, d)] + f[(a, b, d)] + f[(a, b), (c, d)] + f[(a, b, c)]
g3: 4*f[(a, b, c, d)]


In [10]:
show(r"L_{C_4}", L_cycle)
for grade in range(5):
    show(rf"(M_{{C_4}})_{{{grade}}}", M_cycle.extract_grade(grade))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [11]:
cycle_edges = (f_ab, f_bc, f_cd, f_ad)
tree_sum = Polyform.zero()
for omitted in range(len(cycle_edges)):
    tree = unit
    for position, edge in enumerate(cycle_edges):
        if position != omitted:
            tree = tree * edge
    print("Without", cycle_edges[omitted], "->", tree)
    tree_sum = tree_sum + tree
show(r"(M_{C_4})_3", tree_sum)
assert tree_sum.is_equivalent(4 * quForm([(a, b, c, d)]))
assert M_cycle.extract_grade(3).is_equivalent(tree_sum)
assert M_cycle.extract_grade(4).is_zero()

Without f[(a, b)] -> f[(a, b, c, d)]
Without f[(b, c)] -> f[(a, b, c, d)]
Without f[(c, d)] -> f[(a, b, c, d)]
Without f[(a, d)] -> f[(a, b, c, d)]


<IPython.core.display.Math object>

## Коэффициенты при формах

Изменим только проводимость $ab$ на $3/2$. Три дерева содержат это ребро, одно не содержит; их суммарный вес равен $3\cdot3/2+1=11/2$. Сама форма остаётся $[(abcd)]^2$. Здесь используются точные рациональные коэффициенты; символы вершин не заменяются числовыми метками.

In [12]:
L_weighted = Fraction(3, 2) * f_ab + f_bc + f_cd + f_ad
M_weighted = L_weighted.exp()
print(M_weighted)
show(r"(M_{\mathrm{weighted}})_3", M_weighted.extract_grade(3))
assert M_weighted.extract_grade(3).is_equivalent(Fraction(11, 2) * quForm([(a, b, c, d)]))

g0: f[1]
g1: f[(a, d)] + f[(c, d)] + f[(b, c)] + 3/2*f[(a, b)]
g2: f[(a, c, d)] + f[(a, d), (b, c)] + f[(b, c, d)] + 3/2*f[(a, b, d)] + 3/2*f[(a, b), (c, d)] + 3/2*f[(a, b, c)]
g3: 11/2*f[(a, b, c, d)]


<IPython.core.display.Math object>

## Проверки алгебраических равенств

Сравниваются сами полиформы, а не только их нормы или суммы коэффициентов. Сумма коэффициентов дополнительно проверяется именно для полученного лесного разложения; при произвольной эквивалентной перезаписи этот смысл не гарантирован.

In [13]:
for L, M, edges, weights in [
    (L_path, M_path, (f_ab, f_bc, f_cd, f_de), (1, 4, 6, 4, 1)),
    (L_cycle, M_cycle, cycle_edges, (1, 4, 6, 4, 0)),
]:
    product = unit
    for edge in edges:
        product = product * (unit + edge)
    assert product.is_equivalent(M)
    for grade in range(5):
        part = M.extract_grade(grade)
        assert part.is_equivalent(Fraction(1, factorial(grade)) * (L ** grade))
        # Valid for this forest expansion, not arbitrary equivalent rewrites.
        assert sum(part.terms.values()) == weights[grade]
assert M_path.extract_grade(4).is_equivalent(quForm([(a, b, c, d, e)]).as_polyform())
print("All algebraic checks passed.")

All algebraic checks passed.

## От формы к числу

Это отдельный, необязательный шаг: `potential` и `norm` оценивают построенные формы. Они не заменяют символьного разложения.

In [14]:
print("tau(P5) =", M_path.potential(quForm.one()))
print("tau(C4) =", M_cycle.potential(quForm.one()))
print("R_ae(P5) =", M_path.norm(quForm([(a, e)])))
assert M_path.potential(quForm.one()) == 1
assert M_cycle.potential(quForm.one()) == 4
assert M_path.norm(quForm([(a, e)])) == 4

tau(P5) = 1
tau(C4) = 4
R_ae(P5) = 4.0


## Попробуйте изменить

Замените проводимость $ab$ на 2 и просмотрите все грейды: старшая форма цикла получит коэффициент 7. Затем удалите $ad$ из лапласиана цикла: останется путь на четырёх вершинах, и старшая форма будет иметь коэффициент 1.